In [67]:
# Cell 1: Imports and Setup
import os
import json
import random
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import yaml
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

print("Imports completed successfully!")

Imports completed successfully!


In [68]:
   !uv pip  install ultralytics

Using Python 3.11.13 environment at: /usr
Audited 1 package in 105ms


In [69]:
# Cell 2: Configuration
class Config:
    """Configuration for YOLO Object Detection on TACO Dataset"""
    
    # ==================== MODEL SELECTION ====================
    # Options: 'yolov5s', 'yolov5m', 'yolov5l', 'yolov8s', 'yolov8m', 'yolov8l'
    YOLO_MODEL = 'yolov8m'
    
    # ==================== AUGMENTATION ====================
    USE_AUGMENTATION = True  # Set to False for no augmentation
    
    # ==================== PATHS ====================
    # Kaggle paths
    ANNOTATIONS_PATH = '/kaggle/input/tacotrashdataset/data/annotations.json'
    META_CSV_PATH = '/kaggle/input/tacotrashdataset/meta_df.csv'
    IMAGES_ROOT = '/kaggle/input/tacotrashdataset/data'
    OUTPUT_DIR = '/kaggle/working'
    YOLO_DATASET_DIR = '/kaggle/working/yolo_dataset'
    
    # ==================== TOP 5 CLASSES ====================
    # Original TACO category IDs -> YOLO class indices (0-indexed)
    TOP5_CLASSES = {
        59: 0,  # Cigarette
        58: 1,  # Unlabeled litter
        36: 2,  # Plastic film
        5:  3,   # Clear plastic bottle
        29: 4,  # Other plastic
    }
    TOP5_CLASS_NAMES = ['Cigarette', 'Unlabeled litter', 'Plastic film', 
                        'Clear plastic bottle', 'Other plastic']
    NUM_CLASSES = 5
    
    # ==================== TRAINING PARAMETERS ====================
    
    # CRITICAL: Large image size for small object detection (cigarettes!)
    IMAGE_SIZE = 1280  # *** KEY CHANGE: 640 -> 1280 ***
    BATCH_SIZE = 4     # Reduced for 1280 images on T4 (16GB)
    NUM_EPOCHS = 60
    LEARNING_RATE = 0.001  # Lower LR for stability
    
    # Data split
    TRAIN_SPLIT = 0.8
    VAL_SPLIT = 0.1
    TEST_SPLIT = 0.1
    
    # Early stopping patience
    PATIENCE = 20
    
    # ==================== AUGMENTATION SETTINGS ====================
    # GENTLE augmentations - preserve small objects like cigarettes
    AUG_SETTINGS = {
        'hsv_h': 0.015,
        'hsv_s': 0.4,        # Reduced
        'hsv_v': 0.3,        # Reduced
        'degrees': 5.0,      # Reduced from 10 - small objects sensitive
        'translate': 0.05,   # Reduced - prevents cropping small objects
        'scale': 0.2,        # Reduced from 0.5 - critical for small objects
        'shear': 2.0,        # Reduced
        'perspective': 0.0,  # Disabled
        'flipud': 0.0,       # Disabled
        'fliplr': 0.5,
        'mosaic': 0.3,       # Reduced from 1.0 - mosaic shrinks images!
        'mixup': 0.0,        # Disabled - obscures small objects
        'copy_paste': 0.3,   # Increased - helps minority class
    }
    
    # No augmentation settings
    NO_AUG_SETTINGS = {
        'hsv_h': 0.0,
        'hsv_s': 0.0,
        'hsv_v': 0.0,
        'degrees': 0.0,
        'translate': 0.0,
        'scale': 0.0,
        'shear': 0.0,
        'perspective': 0.0,
        'flipud': 0.0,
        'fliplr': 0.0,
        'mosaic': 0.0,
        'mixup': 0.0,
        'copy_paste': 0.0,
    }
    # No augmentation settings
    NO_AUG_SETTINGS = {
        'hsv_h': 0.0,
        'hsv_s': 0.0,
        'hsv_v': 0.0,
        'degrees': 0.0,
        'translate': 0.0,
        'scale': 0.0,
        'shear': 0.0,
        'perspective': 0.0,
        'flipud': 0.0,
        'fliplr': 0.0,
        'mosaic': 0.0,
        'mixup': 0.0,
        'copy_paste': 0.0,
    }
    
    @classmethod
    def get_aug_settings(cls):
        """Return augmentation settings based on USE_AUGMENTATION flag"""
        return cls.AUG_SETTINGS if cls.USE_AUGMENTATION else cls.NO_AUG_SETTINGS
    
    @classmethod
    def get_model_family(cls):
        """Return 'yolov5' or 'yolov8' based on selected model"""
        return 'yolov5' if 'yolov5' in cls.YOLO_MODEL else 'yolov8'
    
    @classmethod
    def print_config(cls):
        """Print current configuration"""
        print("="*60)
        print("YOLO Object Detection Configuration")
        print("="*60)
        print(f"Model: {cls.YOLO_MODEL}")
        print(f"Model Family: {cls.get_model_family()}")
        print(f"Augmentation: {'Enabled' if cls.USE_AUGMENTATION else 'Disabled'}")
        print(f"Image Size: {cls.IMAGE_SIZE}")
        print(f"Batch Size: {cls.BATCH_SIZE}")
        print(f"Epochs: {cls.NUM_EPOCHS}")
        print(f"Number of Classes: {cls.NUM_CLASSES}")
        print(f"Classes: {cls.TOP5_CLASS_NAMES}")
        print("="*60)

config = Config()
config.print_config()

YOLO Object Detection Configuration
Model: yolov8m
Model Family: yolov8
Augmentation: Enabled
Image Size: 1280
Batch Size: 4
Epochs: 60
Number of Classes: 5
Classes: ['Cigarette', 'Unlabeled litter', 'Plastic film', 'Clear plastic bottle', 'Other plastic']


In [70]:
# # Cell 3: Install YOLO (YOLOv5 or YOLOv8 based on config)
# import subprocess
# import sys

# def install_yolo():
#     """Install appropriate YOLO version based on config"""
#     model_family = config.get_model_family()
    
#     if model_family == 'yolov8':
#         print("Installing Ultralytics (YOLOv8)...")
#         subprocess.check_call([sys.executable, -m, 'uv', 'pip', 'install', 'ultralytics'])
#         print("YOLOv8 installed successfully!")
#     else:
#         print("Installing YOLOv5...")
#         # Clone YOLOv5 repo if not exists
#         if not os.path.exists('/kaggle/working/yolov5'):
#             subprocess.check_call(['git', 'clone', 'https://github.com/ultralytics/yolov5.git', '/kaggle/working/yolov5'])
#         subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/kaggle/working/yolov5/requirements.txt'])
#         print("YOLOv5 installed successfully!")

# install_yolo()

In [71]:
# Cell 4: Data Loading and Filtering Functions
def load_and_filter_data(annotations_path, meta_csv_path, top5_classes):
    """
    Load original annotations and filter using meta_df for top 5 classes.
    
    Args:
        annotations_path: Path to original TACO annotations.json
        meta_csv_path: Path to meta_df.csv
        top5_classes: Dict mapping original category_id to YOLO class index
    
    Returns:
        filtered_data: Dict with images, annotations, categories
    """
    # Load meta_df to get annotation IDs for top 5 classes
    print("Loading meta_df.csv...")
    meta_df = pd.read_csv(meta_csv_path)
    
    # Filter for top 5 classes by category name
    top5_names = config.TOP5_CLASS_NAMES
    filtered_meta = meta_df[meta_df['cat_name'].isin(top5_names)]
    
    print(f"Found {len(filtered_meta)} annotations for top 5 classes")
    print("\nClass distribution:")
    print(filtered_meta['cat_name'].value_counts())
    
    # Get annotation IDs and image IDs
    valid_ann_ids = set(filtered_meta['ann_id'].tolist())
    valid_img_ids = set(filtered_meta['img_id'].tolist())
    
    # Load original annotations
    print(f"\nLoading original annotations from {annotations_path}...")
    with open(annotations_path, 'r') as f:
        original_data = json.load(f)
    
    print(f"Original annotations: {len(original_data['annotations'])}")
    print(f"Original images: {len(original_data['images'])}")
    
    # Filter annotations by annotation ID and remap category IDs
    filtered_annotations = []
    for ann in original_data['annotations']:
        if ann['id'] in valid_ann_ids:
            original_cat_id = ann['category_id']
            if original_cat_id in top5_classes:
                new_ann = ann.copy()
                new_ann['category_id'] = top5_classes[original_cat_id]  # 0-indexed for YOLO
                filtered_annotations.append(new_ann)
    
    print(f"Filtered annotations: {len(filtered_annotations)}")
    
    # Filter images to only those with valid annotations
    filtered_images = [img for img in original_data['images'] if img['id'] in valid_img_ids]
    print(f"Filtered images: {len(filtered_images)}")
    
    # Create new categories list for top 5
    new_categories = [
        {'id': i, 'name': name} for i, name in enumerate(config.TOP5_CLASS_NAMES)
    ]
    
    filtered_data = {
        'images': filtered_images,
        'annotations': filtered_annotations,
        'categories': new_categories
    }
    
    return filtered_data

print("Data loading functions defined.")

Data loading functions defined.


In [72]:
# Cell 5: YOLO Dataset Preparation Functions
def bbox_coco_to_yolo(bbox, img_width, img_height):
    """
    Convert COCO bbox format to YOLO format.
    COCO: [x_min, y_min, width, height] (absolute)
    YOLO: [x_center, y_center, width, height] (normalized 0-1)
    """
    x_min, y_min, width, height = bbox
    
    # Calculate center coordinates
    x_center = (x_min + width / 2) / img_width
    y_center = (y_min + height / 2) / img_height
    
    # Normalize width and height
    norm_width = width / img_width
    norm_height = height / img_height
    
    # Clip to valid range
    x_center = max(0, min(1, x_center))
    y_center = max(0, min(1, y_center))
    norm_width = max(0, min(1, norm_width))
    norm_height = max(0, min(1, norm_height))
    
    return [x_center, y_center, norm_width, norm_height]


def polygon_to_bbox(segmentation):
    """
    Convert polygon segmentation to bounding box.
    Returns: [x_min, y_min, width, height] in COCO format
    """
    all_x = []
    all_y = []
    
    for seg in segmentation:
        # Reshape to (N, 2)
        points = np.array(seg).reshape(-1, 2)
        all_x.extend(points[:, 0])
        all_y.extend(points[:, 1])
    
    x_min = min(all_x)
    y_min = min(all_y)
    x_max = max(all_x)
    y_max = max(all_y)
    
    width = x_max - x_min
    height = y_max - y_min
    
    return [x_min, y_min, width, height]


def create_yolo_dataset(filtered_data, images_root, output_dir, split_ids):
    """
    Create YOLO format dataset structure.
    
    Structure:
    output_dir/
        images/
            train/
            val/
            test/
        labels/
            train/
            val/
            test/
    """
    # Create directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(output_dir, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(output_dir, 'labels', split), exist_ok=True)
    
    # Create lookup dictionaries
    images_dict = {img['id']: img for img in filtered_data['images']}
    
    # Group annotations by image_id
    annotations_by_image = defaultdict(list)
    for ann in filtered_data['annotations']:
        annotations_by_image[ann['image_id']].append(ann)
    
    stats = {'train': 0, 'val': 0, 'test': 0}
    
    for split, img_ids in split_ids.items():
        print(f"\nProcessing {split} split ({len(img_ids)} images)...")
        
        for img_id in tqdm(img_ids, desc=split):
            if img_id not in images_dict:
                continue
            
            img_info = images_dict[img_id]
            img_width = img_info['width']
            img_height = img_info['height']
            
            # Copy image
            src_path = os.path.join(images_root, img_info['file_name'])
            if not os.path.exists(src_path):
                continue
            
            # Create unique filename
            img_filename = f"{img_id:06d}.jpg"
            dst_path = os.path.join(output_dir, 'images', split, img_filename)
            shutil.copy(src_path, dst_path)
            
            # Create label file
            label_filename = f"{img_id:06d}.txt"
            label_path = os.path.join(output_dir, 'labels', split, label_filename)
            
            annotations = annotations_by_image.get(img_id, [])
            
            with open(label_path, 'w') as f:
                for ann in annotations:
                    class_id = ann['category_id']
                    
                    # Get bbox - prefer 'bbox' field, fall back to polygon
                    if 'bbox' in ann and ann['bbox']:
                        bbox_coco = ann['bbox']
                    elif 'segmentation' in ann and ann['segmentation']:
                        bbox_coco = polygon_to_bbox(ann['segmentation'])
                    else:
                        continue
                    
                    # Convert to YOLO format
                    bbox_yolo = bbox_coco_to_yolo(bbox_coco, img_width, img_height)
                    
                    # Write YOLO format: class_id x_center y_center width height
                    f.write(f"{class_id} {bbox_yolo[0]:.6f} {bbox_yolo[1]:.6f} {bbox_yolo[2]:.6f} {bbox_yolo[3]:.6f}\n")
            
            stats[split] += 1
    
    print(f"\nDataset creation complete!")
    print(f"  Train: {stats['train']} images")
    print(f"  Val: {stats['val']} images")
    print(f"  Test: {stats['test']} images")
    
    return stats

print("YOLO dataset preparation functions defined.")

YOLO dataset preparation functions defined.


In [73]:
# Cell 6: Create YAML Configuration for YOLO
def create_yolo_yaml(output_dir, class_names):
    """
    Create YAML configuration file for YOLO training.
    """
    yaml_content = {
        'path': output_dir,
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': len(class_names),
        'names': class_names
    }
    
    yaml_path = os.path.join(output_dir, 'dataset.yaml')
    
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)
    
    print(f"Created YAML config at: {yaml_path}")
    print("\nYAML content:")
    print(yaml.dump(yaml_content, default_flow_style=False, sort_keys=False))
    
    return yaml_path

print("YAML creation function defined.")

YAML creation function defined.


In [74]:
# Cell 7: Load and Prepare Dataset
print("="*60)
print("Loading and Preparing YOLO Dataset")
print("="*60)

# Load and filter data
filtered_data = load_and_filter_data(
    config.ANNOTATIONS_PATH,
    config.META_CSV_PATH,
    config.TOP5_CLASSES
)

# Get all image IDs with annotations
image_ids_with_annotations = list(set(
    ann['image_id'] for ann in filtered_data['annotations']
))
print(f"\nTotal images with annotations: {len(image_ids_with_annotations)}")

# Shuffle and split
random.shuffle(image_ids_with_annotations)

n_total = len(image_ids_with_annotations)
n_train = int(n_total * config.TRAIN_SPLIT)
n_val = int(n_total * config.VAL_SPLIT)

split_ids = {
    'train': image_ids_with_annotations[:n_train],
    'val': image_ids_with_annotations[n_train:n_train + n_val],
    'test': image_ids_with_annotations[n_train + n_val:]
}

print(f"\nData split:")
print(f"  Train: {len(split_ids['train'])} images")
print(f"  Val: {len(split_ids['val'])} images")
print(f"  Test: {len(split_ids['test'])} images")

# Create YOLO dataset
print("\n" + "="*60)
print("Creating YOLO Format Dataset")
print("="*60)

stats = create_yolo_dataset(
    filtered_data,
    config.IMAGES_ROOT,
    config.YOLO_DATASET_DIR,
    split_ids
)

# Create YAML config
yaml_path = create_yolo_yaml(config.YOLO_DATASET_DIR, config.TOP5_CLASS_NAMES)

Loading and Preparing YOLO Dataset
Loading meta_df.csv...
Found 2193 annotations for top 5 classes

Class distribution:
cat_name
Cigarette               667
Unlabeled litter        517
Plastic film            451
Clear plastic bottle    285
Other plastic           273
Name: count, dtype: int64

Loading original annotations from /kaggle/input/tacotrashdataset/data/annotations.json...
Original annotations: 4784
Original images: 1500
Filtered annotations: 2193
Filtered images: 861

Total images with annotations: 861

Data split:
  Train: 688 images
  Val: 86 images
  Test: 87 images

Creating YOLO Format Dataset

Processing train split (688 images)...


train: 100%|██████████| 688/688 [00:02<00:00, 241.73it/s]



Processing val split (86 images)...


val: 100%|██████████| 86/86 [00:00<00:00, 248.23it/s]



Processing test split (87 images)...


test: 100%|██████████| 87/87 [00:00<00:00, 251.16it/s]


Dataset creation complete!
  Train: 688 images
  Val: 86 images
  Test: 87 images
Created YAML config at: /kaggle/working/yolo_dataset/dataset.yaml

YAML content:
path: /kaggle/working/yolo_dataset
train: images/train
val: images/val
test: images/test
nc: 5
names:
- Cigarette
- Unlabeled litter
- Plastic film
- Clear plastic bottle
- Other plastic



In [75]:
# Cell 8: Visualize Sample Data
def visualize_yolo_samples(dataset_dir, split='train', num_samples=4):
    """
    Visualize sample images with bounding boxes from YOLO dataset.
    """
    images_dir = os.path.join(dataset_dir, 'images', split)
    labels_dir = os.path.join(dataset_dir, 'labels', split)
    
    image_files = sorted(os.listdir(images_dir))[:num_samples]
    
    # Colors for each class
    colors = [
        (255, 0, 0),      # Cigarette - Red
        (0, 255, 0),      # Unlabeled litter - Green
        (0, 0, 255),      # Plastic film - Blue
        (255, 255, 0),    # Clear plastic bottle - Yellow
        (255, 0, 255),    # Other plastic - Magenta
    ]
    
    fig, axes = plt.subplots(1, num_samples, figsize=(5*num_samples, 5))
    if num_samples == 1:
        axes = [axes]
    
    for idx, img_file in enumerate(image_files):
        img_path = os.path.join(images_dir, img_file)
        label_path = os.path.join(labels_dir, img_file.replace('.jpg', '.txt'))
        
        # Read image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Read labels and draw boxes
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = int(parts[0])
                        x_center, y_center, bw, bh = map(float, parts[1:5])
                        
                        # Convert to pixel coordinates
                        x1 = int((x_center - bw/2) * w)
                        y1 = int((y_center - bh/2) * h)
                        x2 = int((x_center + bw/2) * w)
                        y2 = int((y_center + bh/2) * h)
                        
                        # Draw rectangle
                        color = colors[class_id % len(colors)]
                        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                        
                        # Add label
                        label = config.TOP5_CLASS_NAMES[class_id]
                        cv2.putText(img, label, (x1, y1-5), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        axes[idx].imshow(img)
        axes[idx].set_title(f'{split}: {img_file}')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR, f'yolo_samples_{split}.png'), dpi=150)
    plt.show()

# Visualize samples
print("Visualizing training samples...")
visualize_yolo_samples(config.YOLO_DATASET_DIR, 'train', num_samples=4)

Visualizing training samples...


In [76]:
# Cell 9: Training Function for YOLOv8
def train_yolov8(yaml_path, model_name, epochs, batch_size, img_size, aug_settings, output_dir):
    """
    Train YOLOv8 model.
    """
    from ultralytics import YOLO
    
    # Load pretrained model
    model_weights = f"{model_name}.pt"
    print(f"Loading pretrained model: {model_weights}")
    model = YOLO(model_weights)
    
    # Training arguments
    train_args = {
        'data': yaml_path,
        'epochs': epochs,
        'batch': batch_size,
        'imgsz': img_size,
        'patience': config.PATIENCE,
        'save': True,
        'project': output_dir,
        'name': f'{model_name}_taco',
        'exist_ok': True,
        'pretrained': True,
        'optimizer': 'AdamW',
        'lr0': config.LEARNING_RATE,
        'weight_decay': 0.0005,
        'warmup_epochs': 3,
        'close_mosaic': 10,
        'amp': True,  # Mixed precision
        'verbose': True,
    }
    
    # Add augmentation settings
    train_args.update(aug_settings)
    
    print("\nTraining configuration:")
    for key, value in train_args.items():
        print(f"  {key}: {value}")
    
    # Train
    print("\n" + "="*60)
    print("Starting Training")
    print("="*60 + "\n")
    
    results = model.train(**train_args)
    
    return model, results

print("YOLOv8 training function defined.")

YOLOv8 training function defined.


In [77]:
# Cell 10: Training Function for YOLOv5
def train_yolov5(yaml_path, model_name, epochs, batch_size, img_size, aug_settings, output_dir):
    """
    Train YOLOv5 model.
    """
    import sys
    sys.path.insert(0, '/kaggle/working/yolov5')
    
    from yolov5 import train as yolov5_train
    
    # Model weights
    model_weights = f"{model_name}.pt"
    print(f"Using pretrained model: {model_weights}")
    
    # Create hyperparameters file with augmentation settings
    hyp_path = os.path.join(output_dir, 'hyp_custom.yaml')
    
    hyp = {
        'lr0': config.LEARNING_RATE,
        'lrf': 0.01,
        'momentum': 0.937,
        'weight_decay': 0.0005,
        'warmup_epochs': 3.0,
        'warmup_momentum': 0.8,
        'warmup_bias_lr': 0.1,
        'box': 0.05,
        'cls': 0.5,
        'cls_pw': 1.0,
        'obj': 1.0,
        'obj_pw': 1.0,
        'iou_t': 0.20,
        'anchor_t': 4.0,
        'fl_gamma': 0.0,
    }
    hyp.update(aug_settings)
    
    with open(hyp_path, 'w') as f:
        yaml.dump(hyp, f)
    
    print("\nTraining YOLOv5...")
    print(f"  Model: {model_name}")
    print(f"  Epochs: {epochs}")
    print(f"  Batch size: {batch_size}")
    print(f"  Image size: {img_size}")
    
    # Train using command line (more reliable)
    cmd = f"""python /kaggle/working/yolov5/train.py \
        --weights {model_weights} \
        --data {yaml_path} \
        --epochs {epochs} \
        --batch-size {batch_size} \
        --img {img_size} \
        --hyp {hyp_path} \
        --project {output_dir} \
        --name {model_name}_taco \
        --exist-ok \
        --patience {config.PATIENCE}"""
    
    print(f"\nRunning command:\n{cmd}")
    os.system(cmd)
    
    return None, None

print("YOLOv5 training function defined.")

YOLOv5 training function defined.


In [78]:
# Cell 11: Main Training Execution
print("="*60)
print(f"Starting YOLO Training")
print("="*60)
config.print_config()

# Get augmentation settings
aug_settings = config.get_aug_settings()
print(f"\nAugmentation settings:")
for key, value in aug_settings.items():
    print(f"  {key}: {value}")

# Train based on model family
if config.get_model_family() == 'yolov8':
    model, results = train_yolov8(
        yaml_path=yaml_path,
        model_name=config.YOLO_MODEL,
        epochs=config.NUM_EPOCHS,
        batch_size=config.BATCH_SIZE,
        img_size=config.IMAGE_SIZE,
        aug_settings=aug_settings,
        output_dir=config.OUTPUT_DIR
    )
else:
    model, results = train_yolov5(
        yaml_path=yaml_path,
        model_name=config.YOLO_MODEL,
        epochs=config.NUM_EPOCHS,
        batch_size=config.BATCH_SIZE,
        img_size=config.IMAGE_SIZE,
        aug_settings=aug_settings,
        output_dir=config.OUTPUT_DIR
    )

print("\n" + "="*60)
print("Training Complete!")
print("="*60)

Starting YOLO Training
YOLO Object Detection Configuration
Model: yolov8m
Model Family: yolov8
Augmentation: Enabled
Image Size: 1280
Batch Size: 4
Epochs: 60
Number of Classes: 5
Classes: ['Cigarette', 'Unlabeled litter', 'Plastic film', 'Clear plastic bottle', 'Other plastic']

Augmentation settings:
  hsv_h: 0.015
  hsv_s: 0.4
  hsv_v: 0.3
  degrees: 5.0
  translate: 0.05
  scale: 0.2
  shear: 2.0
  perspective: 0.0
  flipud: 0.0
  fliplr: 0.5
  mosaic: 0.3
  mixup: 0.0
  copy_paste: 0.3
Loading pretrained model: yolov8m.pt

Training configuration:
  data: /kaggle/working/yolo_dataset/dataset.yaml
  epochs: 60
  batch: 4
  imgsz: 1280
  patience: 20
  save: True
  project: /kaggle/working
  name: yolov8m_taco
  exist_ok: True
  pretrained: True
  optimizer: AdamW
  lr0: 0.001
  weight_decay: 0.0005
  warmup_epochs: 3
  close_mosaic: 10
  amp: True
  verbose: True
  hsv_h: 0.015
  hsv_s: 0.4
  hsv_v: 0.3
  degrees: 5.0
  translate: 0.05
  scale: 0.2
  shear: 2.0
  perspective: 0.0
  

In [79]:
# Cell 12: Evaluate on Test Set (YOLOv8)
def evaluate_yolov8(model_path, yaml_path, img_size, output_dir):
    """
    Evaluate YOLOv8 model on test set.
    """
    from ultralytics import YOLO
    
    print("Loading best model for evaluation...")
    model = YOLO(model_path)
    
    # Validate on test set
    print("\nEvaluating on test set...")
    results = model.val(
        data=yaml_path,
        split='test',
        imgsz=img_size,
        batch=16,
        save_json=True,
        project=output_dir,
        name='test_evaluation',
        exist_ok=True
    )
    
    # Print results
    print("\n" + "="*60)
    print("Test Set Results")
    print("="*60)
    print(f"  mAP@0.5: {results.box.map50:.4f}")
    print(f"  mAP@0.5:0.95: {results.box.map:.4f}")
    print(f"  Precision: {results.box.mp:.4f}")
    print(f"  Recall: {results.box.mr:.4f}")
    
    # Per-class results
    print("\nPer-class AP@0.5:")
    for i, name in enumerate(config.TOP5_CLASS_NAMES):
        if i < len(results.box.ap50):
            print(f"  {name}: {results.box.ap50[i]:.4f}")
    
    return results

# Find and evaluate best model
if config.get_model_family() == 'yolov8':
    best_model_path = os.path.join(config.OUTPUT_DIR, f'{config.YOLO_MODEL}_taco', 'weights', 'best.pt')
    if os.path.exists(best_model_path):
        test_results = evaluate_yolov8(
            model_path=best_model_path,
            yaml_path=yaml_path,
            img_size=config.IMAGE_SIZE,
            output_dir=config.OUTPUT_DIR
        )
    else:
        print(f"Best model not found at: {best_model_path}")

Loading best model for evaluation...

Evaluating on test set...
Ultralytics 8.3.233 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,842,655 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3220.8±553.7 MB/s, size: 2347.4 KB)
val: Scanning /kaggle/working/yolo_dataset/labels/test.cache... 87 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 87/87 190.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.7s/it 10.4s2.0s
                   all         87        229      0.279      0.311      0.253      0.157
             Cigarette         22        105      0.502      0.286      0.298      0.164
      Unlabeled litter         29         47      0.195      0.234      0.148     0.0836
          Plastic film         29         35       0.19      0.295       0.16     0.0891
  Clear plastic bottle         26         29      0.424   

In [80]:
# Cell 13: Inference and Visualization
def run_inference_and_visualize(model_path, test_images_dir, output_dir, num_samples=8):
    """
    Run inference on test images and visualize results.
    """
    from ultralytics import YOLO
    
    print("Loading model for inference...")
    model = YOLO(model_path)
    
    # Get test images
    test_images = sorted(os.listdir(test_images_dir))[:num_samples]
    test_image_paths = [os.path.join(test_images_dir, img) for img in test_images]
    
    print(f"Running inference on {len(test_image_paths)} images...")
    
    # Run inference
    results = model.predict(
        source=test_image_paths,
        save=True,
        project=output_dir,
        name='predictions',
        exist_ok=True,
        conf=0.25,
        iou=0.45
    )
    
    # Display results
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for idx, result in enumerate(results[:8]):
        if idx >= len(axes):
            break
        
        # Get annotated image
        annotated = result.plot()
        annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        
        axes[idx].imshow(annotated)
        axes[idx].set_title(f'Image {idx+1}', fontsize=10)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'inference_results.png'), dpi=150)
    plt.show()
    
    print(f"\nInference results saved to {output_dir}/predictions/")
    
    return results

# Run inference
if config.get_model_family() == 'yolov8':
    best_model_path = os.path.join(config.OUTPUT_DIR, f'{config.YOLO_MODEL}_taco', 'weights', 'best.pt')
    test_images_dir = os.path.join(config.YOLO_DATASET_DIR, 'images', 'test')
    
    if os.path.exists(best_model_path):
        inference_results = run_inference_and_visualize(
            model_path=best_model_path,
            test_images_dir=test_images_dir,
            output_dir=config.OUTPUT_DIR,
            num_samples=8
        )

Loading model for inference...
Running inference on 8 images...

0: 1280x1280 (no detections), 97.8ms
1: 1280x1280 1 Plastic film, 97.8ms
2: 1280x1280 (no detections), 97.8ms
3: 1280x1280 (no detections), 97.8ms
4: 1280x1280 1 Unlabeled litter, 2 Plastic films, 1 Clear plastic bottle, 2 Other plastics, 97.8ms
5: 1280x1280 1 Unlabeled litter, 97.8ms
6: 1280x1280 (no detections), 97.8ms
7: 1280x1280 (no detections), 97.8ms
Speed: 10.7ms preprocess, 97.8ms inference, 0.4ms postprocess per image at shape (1, 3, 1280, 1280)
Results saved to /kaggle/working/predictions

Inference results saved to /kaggle/working/predictions/


In [81]:
# Cell 14: Summary and Model Export
print("="*60)
print("Training Summary")
print("="*60)

config.print_config()

# List saved files
print("\nSaved files:")
model_dir = os.path.join(config.OUTPUT_DIR, f'{config.YOLO_MODEL}_taco')
if os.path.exists(model_dir):
    for root, dirs, files in os.walk(model_dir):
        level = root.replace(model_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:10]:  # Limit files shown
            print(f'{subindent}{file}')
        if len(files) > 10:
            print(f'{subindent}... and {len(files)-10} more files')

# Export model to different formats (optional)
print("\n" + "="*60)
print("Model Export Options")
print("="*60)
print("To export the model to different formats, use:")
print("  model.export(format='onnx')  # ONNX format")
print("  model.export(format='torchscript')  # TorchScript")
print("  model.export(format='tflite')  # TensorFlow Lite")

print("\n" + "="*60)
print("YOLO Object Detection Pipeline Complete!")
print("="*60)

Training Summary
YOLO Object Detection Configuration
Model: yolov8m
Model Family: yolov8
Augmentation: Enabled
Image Size: 1280
Batch Size: 4
Epochs: 60
Number of Classes: 5
Classes: ['Cigarette', 'Unlabeled litter', 'Plastic film', 'Clear plastic bottle', 'Other plastic']

Saved files:
yolov8m_taco/
  val_batch2_pred.jpg
  BoxP_curve.png
  results.png
  train_batch8600.jpg
  train_batch0.jpg
  train_batch1.jpg
  train_batch8602.jpg
  confusion_matrix.png
  train_batch2.jpg
  val_batch0_pred.jpg
  ... and 18 more files
  weights/
    last.pt
    best.pt

Model Export Options
To export the model to different formats, use:
  model.export(format='onnx')  # ONNX format
  model.export(format='torchscript')  # TorchScript
  model.export(format='tflite')  # TensorFlow Lite

YOLO Object Detection Pipeline Complete!


In [82]:
import shutil

# Define the source directory and the output zip file name
# Note: Do not include the .zip extension in the output filename
source_dir = '/kaggle/working/test_evaluation' 
output_zip_name = 'test_evaluation'

# Create the zip archive
# The archive file will be named 'my_archive.zip' in the current directory
shutil.make_archive(output_zip_name, 'zip', source_dir)

print(f"Folder '{source_dir}' successfully zipped to '{output_zip_name}.zip'")

from IPython.display import FileLink
FileLink(r'test_evaluation.zip') 



Folder '/kaggle/working/test_evaluation' successfully zipped to 'test_evaluation.zip'


/kaggle/working/test_evaluation.zip

In [83]:
import shutil

# Define the source directory and the output zip file name
# Note: Do not include the .zip extension in the output filename
source_dir = '/kaggle/working/predictions' 
output_zip_name = 'predictions'

# Create the zip archive
# The archive file will be named 'my_archive.zip' in the current directory
shutil.make_archive(output_zip_name, 'zip', source_dir)

print(f"Folder '{source_dir}' successfully zipped to '{output_zip_name}.zip'")

from IPython.display import FileLink
FileLink(r'predictions.zip') 

Folder '/kaggle/working/predictions' successfully zipped to 'predictions.zip'


/kaggle/working/predictions.zip

In [84]:
import shutil

# Define the source directory and the output zip file name
# Note: Do not include the .zip extension in the output filename
source_dir = '/kaggle/working/yolov8m_taco' 
output_zip_name = 'yolov8m_taco'

# Create the zip archive
# The archive file will be named 'my_archive.zip' in the current directory
shutil.make_archive(output_zip_name, 'zip', source_dir)

print(f"Folder '{source_dir}' successfully zipped to '{output_zip_name}.zip'")

from IPython.display import FileLink
FileLink(r'yolov8m_taco.zip') 

Folder '/kaggle/working/yolov8m_taco' successfully zipped to 'yolov8m_taco.zip'


/kaggle/working/yolov8m_taco.zip